In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)
N_NORMAL, N_FRAUD = 2000, 100

normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})

fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [3]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(clf, f)
print("Model zapisany do fraud_model.pkl")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Model zapisany do fraud_model.pkl


In [4]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    proba = model.predict_proba(X)[0, 1]
    return {"is_fraud": bool(proba >= 0.5), "fraud_probability": round(float(proba), 4)}

@app.get("/health")
def health():
    return {"status": "ok"}

Overwriting fraud_api.py


In [5]:
import requests

r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Normalna:", r.json())

r = requests.post("http://localhost:8001/score",
    json={"amount": 5500, "is_electronics": 1, "tx_per_minute": 12})
print("Podejrzana:", r.json())

r = requests.get("http://localhost:8001/health")
print("Health:", r.json())

Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 0.99}
Health: {'status': 'ok'}


In [6]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

for message in consumer:
    tx = message.value
    is_elec = 1 if tx.get('category') == 'elektronika' else 0
    features = {"amount": tx['amount'], "is_electronics": is_elec, "tx_per_minute": 5}

    try:
        r = requests.post(API_URL, json=features, timeout=2)
        result = r.json()
        if result['is_fraud']:
            tx['fraud_probability'] = result['fraud_probability']
            tx['alert_source'] = 'ml_model'
            alert_producer.send('alerts', value=tx)
            print(f"🚨 FRAUD [{result['fraud_probability']:.0%}] {tx['tx_id']} | {tx['amount']:.2f} PLN")
            alert_producer.flush()
        else:
            print(f"   OK    | {tx['tx_id']} | {tx['amount']:.2f} PLN")
    except Exception as e:
        print(f"Błąd API: {e}")

Overwriting ml_consumer.py
